In [1]:
import os
from sentence_transformers import SentenceTransformer
import faiss

In [2]:
documents = [
    "The weather in New York is cloudy.",
    "The secret password is 'BANANA'.", # Our target
    "Apples are usually red or green.",
    "The capital of France is Paris.",
    "Python is a popular programming language.",
    "The moon orbits the Earth.",
    "Deep learning is a subset of AI.",
    "Water boils at 100 degrees Celsius.",
    "The Great Wall of China is visible from space.",
    "Mount Everest is the tallest mountain."
]

In [3]:
# embedding model
embedding_model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B')

docs_embeddings = embedding_model.encode(documents, convert_to_numpy=True)

docs_embeddings[0]

array([-0.02539867,  0.01325228, -0.00546455, ...,  0.04256301,
        0.03717346,  0.01400935], shape=(1024,), dtype=float32)

In [4]:
dimensions = docs_embeddings.shape[1]

# index for FAISS
index = faiss.IndexFlatL2(dimensions)
index.add(docs_embeddings)

In [5]:
query = "What is the secret password?"

In [6]:
# retrive documents

query_embedding = embedding_model.encode([query])

top_k = 10
distance, indices = index.search(query_embedding, top_k)

print(f"distance : {distance}, indices : {indices}")

distance : [[0.34783643 1.2334532  1.2765858  1.2931278  1.3005309  1.4182941
  1.4508327  1.5350623  1.6011132  1.6109822 ]], indices : [[1 5 0 4 8 6 7 2 3 9]]


In [7]:
top_chunks = [documents[i] for i in indices[0]]
top_chunks

["The secret password is 'BANANA'.",
 'The moon orbits the Earth.',
 'The weather in New York is cloudy.',
 'Python is a popular programming language.',
 'The Great Wall of China is visible from space.',
 'Deep learning is a subset of AI.',
 'Water boils at 100 degrees Celsius.',
 'Apples are usually red or green.',
 'The capital of France is Paris.',
 'Mount Everest is the tallest mountain.']

In [ ]:
from

In [10]:
# reciprocal rank fusion function
def rrf_score(results_lists, k=60):
    fused_scores = {}
    for results in results_lists:
        for rank, doc_id in enumerate(results):
            # rank starts at 0, so we add 1 for the formula
            current_rank = rank + 1
            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0

            fused_scores[doc_id] += 1 / (k + current_rank)

    # Sort documents by their score in descending order
    return sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)

In [12]:
ranked_results = rrf_score([top_chunks, top_chunks])
ranked_results

[("The secret password is 'BANANA'.", 0.03278688524590164),
 ('The moon orbits the Earth.', 0.03225806451612903),
 ('The weather in New York is cloudy.', 0.031746031746031744),
 ('Python is a popular programming language.', 0.03125),
 ('The Great Wall of China is visible from space.', 0.03076923076923077),
 ('Deep learning is a subset of AI.', 0.030303030303030304),
 ('Water boils at 100 degrees Celsius.', 0.029850746268656716),
 ('Apples are usually red or green.', 0.029411764705882353),
 ('The capital of France is Paris.', 0.028985507246376812),
 ('Mount Everest is the tallest mountain.', 0.02857142857142857)]